
### Middleware

In [1]:
import os
from  dotenv import load_dotenv
load_dotenv() 


os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [6]:
# summarization middleware

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

# message based summerization middleware
agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model="google_genai:gemini-2.5-flash",
        trigger=("messages",10),
        keep=("messages",4)
    )],
)

In [ ]:
# Run with thread id

config = {
    "configurable": {
        "thread_id": "test-1"
    }
    
    
}

In [7]:
questions = [
    "what is 3 + 3?",
    "what is 4 + 4?",
    "what is 5 + 5?",
    "what is 6 + 6?",
]

for q in questions:
    res = agent.invoke({"messages": [HumanMessage(content=q)]}, config=config)
    print(res)
    print(len(res["messages"]))
    print("--------------------------------------------------")

{'messages': [HumanMessage(content='what is 3 + 3?', additional_kwargs={}, response_metadata={}, id='ad489fd7-7179-41c8-b489-9c31706a5c65'), AIMessage(content='3 + 3 = 6', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a01b90-c980-7ba0-ac41-2f3d1ce29045-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 33, 'total_tokens': 42, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 26}})]}
2
--------------------------------------------------
{'messages': [HumanMessage(content='what is 3 + 3?', additional_kwargs={}, response_metadata={}, id='ad489fd7-7179-41c8-b489-9c31706a5c65'), AIMessage(content='3 + 3 = 6', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a01b90-c980-7ba0-ac4

In [11]:
# summarization middleware

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool


@tool("setup_search",description="setup search tool")
def search(query: str) -> str:
    return f"search results for {query}"

# message based summerization middleware
agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[search],
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model="google_genai:gemini-2.5-flash",
        trigger=("tokens",550),
        keep=("tokens",200)
    )],
)
config = {
    "configurable": {
        "thread_id": "test-2"
    }
}


def count_tokens(messages):
    return sum(len(m.content.split()) for m in messages)